# datacompy vs. LLM-as-Judge for text-2-SQL

This notebook cross-checks the **LLM-as-Judge** correctness labels produced by
`src/experiments/text2sql/eval.py` against an **execution-based** comparison.

Pipeline:
1. Fetch the traces of an MLflow eval run.
2. Keep **only** the SQL-equality-comparison traces — the prediction traces that
   carry the `sql_is_correct` judge feedback (this drops any orphan
   judge-completion traces, which have no assessments).
3. From each such trace extract the **expected** (reference) and **predicted**
   (generated) SQL.
4. Execute both against the DuckDB database and cast the results to DataFrames.
5. Compare the two result sets with **datacompy**.
6. Compare, sample-by-sample, the LLM-as-Judge `Y/N` label against datacompy's
   `match / differ` verdict.

In [18]:
import os
import warnings

import datacompy
import duckdb
import mlflow
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 90)

TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
EXPERIMENT_NAME = os.environ.get("MLFLOW_EXPERIMENT_NAME", "water-text2sql-eval")

# Specific eval run id (logged by eval.py). Leave None to auto-pick the most
# recent run that actually carries text2sql judge traces.
RUN_ID = None

DB_PATH = "../data/water.duckdb"

# Float tolerances for numeric result-set comparison.
ABS_TOL = 1e-9
REL_TOL = 1e-6

mlflow.set_tracking_uri(TRACKING_URI)
client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
assert experiment is not None, f"experiment {EXPERIMENT_NAME!r} not found"
EXPERIMENT_ID = experiment.experiment_id

## 1. Fetch the run and its traces

In [19]:
def fetch_traces(run_id):
    return mlflow.search_traces(
        locations=[EXPERIMENT_ID],
        run_id=run_id,
        max_results=10000,
        return_type="list",
    )


def has_judge_assessment(trace):
    return any(
        a.name == "sql_is_correct" and getattr(a, "feedback", None) is not None
        for a in (trace.info.assessments or [])
    )

RUN_ID = "882416710a754e5a84e9cff889d03435"
traces = fetch_traces(RUN_ID)

run = client.get_run(RUN_ID)
print(f"run_id       : {RUN_ID}")
print(f"run_name     : {run.info.run_name}")
print(f"model        : {run.data.params.get('model')}")
print(f"final_quality: {run.data.metrics.get('final_quality')}")
print(f"total traces : {len(traces)}")

run_id       : 882416710a754e5a84e9cff889d03435
run_name     : openai_alias-qwen36-35b-17-37-06-26
model        : openai/alias-qwen36-35b
final_quality: 0.5
total traces : 101


## 2. Keep only the SQL-equality-comparison traces

Each prediction trace carries:
- a `sql_is_correct` **Feedback** — the LLM-as-Judge `Y/N` value, its rationale,
  and the generated (predicted) SQL in its metadata;
- a `sql` **Expectation** — the reference (expected) SQL.

`extract_comparison` returns `None` for any trace lacking the `sql_is_correct`
feedback, so only the actual comparison traces survive.

In [20]:
def extract_comparison(trace):
    """Pull the LLM-as-Judge SQL-equality comparison off a prediction trace."""
    judge = predicted = expected = question = rationale = argilla = None
    for a in (trace.info.assessments or []):
        if a.name == "sql_is_correct" and getattr(a, "feedback", None) is not None:
            judge = bool(a.feedback.value)
            rationale = a.rationale
            md = a.metadata or {}
            predicted = md.get("generated_sql")
            question = md.get("question")
            argilla = md.get("argilla_link")
        elif a.name == "sql" and getattr(a, "expectation", None) is not None:
            expected = a.expectation.value
    if judge is None:
        return None
    return {
        "trace_id": trace.info.trace_id,
        "question": question,
        "judge_correct": judge,      # LLM-as-Judge Y/N
        "expected_sql": expected,    # reference SQL
        "predicted_sql": predicted,  # generated SQL
        "rationale": rationale,
        "argilla_link": argilla,
    }


records = [r for t in traces if (r := extract_comparison(t)) is not None]
samples = pd.DataFrame(records)
print(f"comparison traces: {len(samples)}")
samples[["question", "judge_correct", "expected_sql", "predicted_sql"]].head()

comparison traces: 100


,question,judge_correct,expected_sql,predicted_sql
0,How did average outflow from the small substrate lysimeters differ between timer-based...,False,"SELECT\n DATE_TRUNC('MONTH', o.timestamp) AS month,\n AVG(o.Zeitlysi_Efflux_x) AS av...","SELECT\n AVG(o.Zeitlysi_Efflux_x) AS avg_timer_based,\n AVG(o.Sensorlysi_Efflux_x) A..."
1,Which roof type had the strongest correlation between daily average corrected surface ...,False,"WITH daily_rad AS (\n SELECT\n DATE_TRUNC('DAY', timestamp) AS day,\n AVG(ED1_T...","WITH daily AS (\n SELECT\n DATE_TRUNC('day', r.timestamp) AS day,\n r.KD_TSFCko..."
2,How did soil moisture below 20% affect the runoff-to-precipitation ratio of the non-ir...,False,"WITH weather_daily AS (\n SELECT\n DATE_TRUNC('DAY', timestamp) AS day,\n SUM(R...","SELECT\n DATE_TRUNC('day', o.timestamp) AS day,\n SUM(o.Extensiv2_Efflux) / SUM(w.Ra..."
3,What was the average daily cooling benefit of the wetland roof relative to the gravel ...,True,"WITH daily_surface AS (\n SELECT\n DATE_TRUNC('DAY', timestamp) AS day,\n AVG(K...","WITH daily_cooling AS (\n SELECT\n CAST(r.timestamp AS DATE) AS day,\n AVG(r.KD..."
4,How many observations registered that the smart irrigated extensive roof have both hig...,True,SELECT\n COUNT(*)\nFROM swc AS s\nJOIN radiation AS r\n ON s.timestamp = r.timestamp...,SELECT COUNT(*)\nFROM swc\nJOIN radiation ON swc.timestamp = radiation.timestamp\nWHER...


## 3. Execute SQL against DuckDB

In [21]:
con = duckdb.connect(DB_PATH, read_only=True)


def run_sql(sql):
    """Execute SQL against DuckDB; return (DataFrame | None, error | None)."""
    if not sql or not sql.strip() or sql.strip().startswith("-- unanswerable"):
        return None, "empty/unanswerable SQL"
    try:
        return con.execute(sql).fetchdf(), None
    except Exception as e:
        return None, str(e)

## 4. datacompy result-set comparison

Reference and generated queries use different column aliases and emit rows in
arbitrary order, so we align columns **by position**, sort both frames on all
columns, and add a positional `__row__` join key — making datacompy compare the
two result sets row-for-row, order-insensitively.

In [22]:
def datacompy_match(expected_df, predicted_df):
    """Order-insensitive result-set comparison. Returns (matches, report)."""
    if expected_df.shape[1] != predicted_df.shape[1]:
        return False, (
            f"column count differs: expected {expected_df.shape[1]}, "
            f"predicted {predicted_df.shape[1]}"
        )
    exp_df = expected_df.copy()
    pred_df = predicted_df.copy()
    pred_df.columns = exp_df.columns  # align by position
    cols = list(exp_df.columns)
    exp_df = exp_df.sort_values(cols).reset_index(drop=True)
    pred_df = pred_df.sort_values(cols).reset_index(drop=True)
    exp_df["__row__"] = range(len(exp_df))
    pred_df["__row__"] = range(len(pred_df))
    cmp = datacompy.PandasCompare(
        exp_df,
        pred_df,
        join_columns="__row__",
        abs_tol=ABS_TOL,
        rel_tol=REL_TOL,
    )
    return cmp.matches(), cmp.report()

## 5. Run the comparison over every sample

In [23]:
rows = []
for r in records:
    exp_df, exp_err = run_sql(r["expected_sql"])
    pred_df, pred_err = run_sql(r["predicted_sql"])
    report = None
    if exp_err or pred_err:
        match = False
        status = f"exec error (expected: {exp_err}; predicted: {pred_err})"
    else:
        try:
            match, report = datacompy_match(exp_df, pred_df)
            status = "ok"
        except Exception as e:
            match = False
            status = f"compare error: {e}"
    rows.append({
        "expected_sql": r["expected_sql"],
        "predicted_sql": r["predicted_sql"],
        "question": r["question"],
        "judge_correct": r["judge_correct"],
        "datacompy_match": match,
        "agree": r["judge_correct"] == match,
        "expected_shape": None if exp_df is None else exp_df.shape,
        "predicted_shape": None if pred_df is None else pred_df.shape,
        "status": status,
        "rationale": r["rationale"],
        "report": report,
    })

comparison = pd.DataFrame(rows)
comparison[["question", "judge_correct", "datacompy_match", "agree", "status"]]

,question,judge_correct,datacompy_match,agree,status
0,How did average outflow from the small substrate lysimeters differ between timer-based...,False,False,True,ok
1,Which roof type had the strongest correlation between daily average corrected surface ...,False,False,True,"exec error (expected: None; predicted: Binder Error: Could not ORDER BY column ""abs(co..."
2,How did soil moisture below 20% affect the runoff-to-precipitation ratio of the non-ir...,False,False,True,ok
3,What was the average daily cooling benefit of the wetland roof relative to the gravel ...,True,True,True,ok
4,How many observations registered that the smart irrigated extensive roof have both hig...,True,True,True,ok
...,...,...,...,...,...
95,"On days with at least 15 mm precipitation, what percentage of precipitation left each ...",False,False,True,ok
96,How did the daily runoff-to-precipitation ratio differ between the gravel roof and the...,False,False,True,ok
97,Which roof type had the lowest median daily outflow on days with at least 15 mm precip...,False,False,True,ok
98,How often did the non-irrigated extensive green roof produce less daily outflow than t...,True,True,True,ok


## 6. Agreement: LLM-as-Judge vs. datacompy

In [24]:
total = len(comparison)
agree = int(comparison["agree"].sum())
print(f"samples      : {total}")
print(f"agreement    : {agree}/{total} ({agree / total:.1%})")
print(f"disagreement : {total - agree}/{total} ({(total - agree) / total:.1%})")
print()
print("Confusion (rows = LLM-as-Judge, cols = datacompy):")
conf = pd.crosstab(
    comparison["judge_correct"].map({True: "judge Y", False: "judge N"}),
    comparison["datacompy_match"].map({True: "dc match", False: "dc differ"}),
)
conf

samples      : 100
agreement    : 78/100 (78.0%)
disagreement : 22/100 (22.0%)

Confusion (rows = LLM-as-Judge, cols = datacompy):


datacompy_match,dc differ,dc match
judge_correct,,
judge N,49,1
judge Y,21,29


### Where they disagree

In [25]:
disagreements = comparison[~comparison["agree"]]
print(f"{len(disagreements)} disagreements\n")
for _, row in disagreements.iterrows():
    judge = "Y" if row["judge_correct"] else "N"
    dc = "match" if row["datacompy_match"] else "differ"
    print("Q:", row["question"])
    print(f"  judge={judge}  datacompy={dc}  "
          f"exp_shape={row['expected_shape']} pred_shape={row['predicted_shape']}")
    print("  status:", row["status"])
    print("  judge rationale:", (row["rationale"] or "")[:220])

    print("  reference SQL:")
    print(row["expected_sql"])
    print("|" * 90)
    print("  predicted SQL:")
    print(row["predicted_sql"])

    print("-" * 90)

22 disagreements

Q: Which roof type had the largest average reduction in corrected surface temperature relative to the gravel roof during days with downward shortwave radiation above 700 W/m²?
  judge=Y  datacompy=differ  exp_shape=(1, 2) pred_shape=(1, 2)
  status: ok
  judge rationale: The generated SQL correctly implements the logic to find the roof type with the largest average reduction in corrected surface temperature relative to the gravel roof under the specified radiation condition. It computes 
  reference SQL:
WITH filtered AS (
  SELECT
    KD_TSFCkorr,
    ED1_TSFCkorr,
    ED2_TSFCkorr,
    SD_TSFCkorr
  FROM radiation
  WHERE
    KD_SWdown > 700 AND NOT KD_TSFCkorr IS NULL
), reductions AS (
  SELECT
    'smart irrigated extensive green roof' AS roof_type,
    AVG(KD_TSFCkorr - ED1_TSFCkorr) AS avg_reduction
  FROM filtered
  UNION ALL
  SELECT
    'non-irrigated extensive green roof',
    AVG(KD_TSFCkorr - ED2_TSFCkorr)
  FROM filtered
  UNION ALL
  SELECT
    'wetland

### Inspect a single datacompy report

Set `idx` to any row to see the full datacompy diff for that sample.

In [26]:
idx = 0
row = comparison.iloc[idx]
print("Q:", row["question"])
print("judge_correct:", row["judge_correct"], "| datacompy_match:", row["datacompy_match"])
print()
print(row["report"] or row["status"])

Q: How did average outflow from the small substrate lysimeters differ between timer-based and threshold-based irrigation during months with precipitation below 30 mm?
judge_correct: False | datacompy_match: False

column count differs: expected 4, predicted 2


## 7. BIRD-style execution accuracy (EX)

The original [BIRD benchmark](https://github.com/AlibabaResearch/DAMO-ConvAI/blob/main/bird/llm/src/evaluation.py)
scores a prediction with **execution accuracy (EX)**: execute both the predicted
and the ground-truth SQL, take their result rows, and award `1` iff the two row
sets are equal —

```python
def execute_sql(predicted_sql, ground_truth, db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute(predicted_sql)
    predicted_res = cursor.fetchall()
    cursor.execute(ground_truth)
    ground_truth_res = cursor.fetchall()
    res = 0
    if set(predicted_res) == set(ground_truth_res):
        res = 1
    return res
```

We reproduce this exactly here against DuckDB. Note how it differs from the
datacompy check above:

- **`set(...)` equality of whole rows** — order of rows is irrelevant, but each
  row is a full tuple, so **column order matters** and duplicate rows collapse.
- **No alias alignment, no sorting, no float tolerance** — values must match
  exactly. This makes BIRD EX *stricter* than the datacompy comparison (which
  aligns columns by position and allows `abs_tol`/`rel_tol`).

In [27]:
def bird_ex_match(expected_sql, predicted_sql):
    """BIRD execution accuracy (EX): set-equality of result rows.

    Faithful port of `execute_sql` from BIRD's evaluation.py — execute both
    queries with DuckDB, `fetchall()` into row tuples, and return
    ``set(predicted_res) == set(ground_truth_res)``. No column alignment,
    no sorting, no float tolerance. Returns (matches, status).
    """
    if (not predicted_sql or not predicted_sql.strip()
            or predicted_sql.strip().startswith("-- unanswerable")):
        return False, "empty/unanswerable predicted SQL"
    if not expected_sql or not expected_sql.strip():
        return False, "empty expected SQL"
    try:
        predicted_res = con.execute(predicted_sql).fetchall()
    except Exception as e:
        return False, f"predicted exec error: {e}"
    try:
        ground_truth_res = con.execute(expected_sql).fetchall()
    except Exception as e:
        return False, f"expected exec error: {e}"
    try:
        return set(predicted_res) == set(ground_truth_res), "ok"
    except TypeError as e:  # unhashable values in a row
        return predicted_res == ground_truth_res, f"unhashable rows: {e}"

In [28]:
# Score every sample with BIRD EX and attach the column to `comparison`.
# `records` and `comparison` share the same order, so we can assign positionally.
bird_results = [bird_ex_match(r["expected_sql"], r["predicted_sql"]) for r in records]
comparison["bird_ex"] = [m for m, _ in bird_results]
comparison["bird_status"] = [s for _, s in bird_results]

bird_correct = int(comparison["bird_ex"].sum())
print(f"samples            : {len(comparison)}")
print(f"BIRD EX = 1        : {bird_correct}/{len(comparison)} "
      f"({bird_correct / len(comparison):.1%})")
comparison[["question", "judge_correct", "datacompy_match", "bird_ex", "bird_status"]]

samples            : 100
BIRD EX = 1        : 29/100 (29.0%)


,question,judge_correct,datacompy_match,bird_ex,bird_status
0,How did average outflow from the small substrate lysimeters differ between timer-based...,False,False,False,ok
1,Which roof type had the strongest correlation between daily average corrected surface ...,False,False,False,"predicted exec error: Binder Error: Could not ORDER BY column ""abs(corr_val)"": add the..."
2,How did soil moisture below 20% affect the runoff-to-precipitation ratio of the non-ir...,False,False,False,ok
3,What was the average daily cooling benefit of the wetland roof relative to the gravel ...,True,True,False,ok
4,How many observations registered that the smart irrigated extensive roof have both hig...,True,True,True,ok
...,...,...,...,...,...
95,"On days with at least 15 mm precipitation, what percentage of precipitation left each ...",False,False,False,ok
96,How did the daily runoff-to-precipitation ratio differ between the gravel roof and the...,False,False,False,ok
97,Which roof type had the lowest median daily outflow on days with at least 15 mm precip...,False,False,False,ok
98,How often did the non-irrigated extensive green roof produce less daily outflow than t...,True,True,True,ok


## 8. Compare all three: LLM-as-Judge vs. datacompy vs. BIRD EX

Three independent verdicts per prediction:

| metric | what it checks | leniency |
| --- | --- | --- |
| **LLM-as-Judge** | semantic equivalence judged by an LLM | most lenient (intent-aware) |
| **datacompy** | result sets, columns aligned by position + sorted, with float tolerance | medium |
| **BIRD EX** | exact `set()`-equality of whole result rows | strictest |

In [29]:
metrics = {
    "LLM-as-Judge": comparison["judge_correct"],
    "datacompy": comparison["datacompy_match"],
    "BIRD EX": comparison["bird_ex"],
}
total = len(comparison)

# Per-metric "correct" rate.
summary = pd.DataFrame({
    "correct": {name: int(col.sum()) for name, col in metrics.items()},
    "rate": {name: col.mean() for name, col in metrics.items()},
})
print("Per-metric correctness:")
print(summary.assign(rate=lambda d: (d["rate"] * 100).round(1).astype(str) + "%"))
print()

# Pairwise agreement (fraction of samples where the two verdicts coincide).
names = list(metrics)
pairwise = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        pairwise.loc[a, b] = (metrics[a] == metrics[b]).mean()
print("Pairwise agreement:")
pairwise.round(3)

Per-metric correctness:
              correct   rate
LLM-as-Judge       50  50.0%
datacompy          30  30.0%
BIRD EX            29  29.0%

Pairwise agreement:


,LLM-as-Judge,datacompy,BIRD EX
LLM-as-Judge,1.00,0.78,0.77
datacompy,0.78,1.00,0.99
BIRD EX,0.77,0.99,1.00


In [30]:
# Joint pattern: how the three verdicts combine across the dataset.
yn = lambda s: s.map({True: "Y", False: "N"})
patterns = (
    pd.DataFrame({
        "LLM-as-Judge": yn(comparison["judge_correct"]),
        "datacompy": yn(comparison["datacompy_match"]),
        "BIRD EX": yn(comparison["bird_ex"]),
    })
    .value_counts()
    .rename("count")
    .reset_index()
    .sort_values("count", ascending=False, ignore_index=True)
)
patterns["pct"] = (patterns["count"] / total * 100).round(1).astype(str) + "%"

all_agree = int((
    (comparison["judge_correct"] == comparison["datacompy_match"])
    & (comparison["datacompy_match"] == comparison["bird_ex"])
).sum())
print(f"all three agree    : {all_agree}/{total} ({all_agree / total:.1%})")
print(f"some disagreement  : {total - all_agree}/{total} "
      f"({(total - all_agree) / total:.1%})")
print()
print("Joint verdict patterns (Y = correct):")
patterns

all three agree    : 77/100 (77.0%)
some disagreement  : 23/100 (23.0%)

Joint verdict patterns (Y = correct):


,LLM-as-Judge,datacompy,BIRD EX,count,pct
0,N,N,N,49,49.0%
1,Y,Y,Y,28,28.0%
2,Y,N,N,21,21.0%
3,N,Y,Y,1,1.0%
4,Y,Y,N,1,1.0%


### Where the three metrics disagree

Samples on which the three verdicts are not unanimous — useful for spotting where
BIRD EX's strictness (exact column order / values) diverges from datacompy's
position-aligned, tolerant comparison, and where both differ from the judge's
semantic call.

In [ ]:
unanimous = (
    (comparison["judge_correct"] == comparison["datacompy_match"])
    & (comparison["datacompy_match"] == comparison["bird_ex"])
)
split = comparison[~unanimous]
print(f"{len(split)} samples with a non-unanimous verdict\n")
for _, row in split.iterrows():
    print("Q:", row["question"])
    print(f"  LLM-as-Judge={'Y' if row['judge_correct'] else 'N'}  "
          f"datacompy={'match' if row['datacompy_match'] else 'differ'}  "
          f"BIRD EX={'1' if row['bird_ex'] else '0'}")
    print(f"  datacompy status: {row['status']}")
    print(f"  BIRD status     : {row['bird_status']}")
    print(f"  exp_shape={row['expected_shape']} pred_shape={row['predicted_shape']}")
    print(f"  judge rationale : {(row['rationale'] or '')[:200]}")
    print("-" * 90)